In [49]:
import random
import numpy as np
import hashlib
import os

In [50]:
# Initialisation
board = [0] * 9
human_pieces = 3
ai_pieces = 3
winning_combinations = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8],
    [0, 3, 6], [1, 4, 7], [2, 5, 8],
    [0, 4, 8], [2, 4, 6]
]

In [51]:
# Paramètres RL
q_table_file = "C:\\Users\\EEIA\\Desktop\\DOFBOT\\MiniMax\\Test_Renforcement\\m_ql\\q_table.npy"
q_table = {}
epsilon = 0  # Pas d'exploration pour tester

In [52]:
# Afficher le plateau
def print_board(state=None):
    if state is None:
        state = board
    symbols = {0: ".", 1: "X", 2: "O"}
    print("Positions : 0 1 2 | 3 4 5 | 6 7 8")
    for i in range(0, 9, 3):
        print(f"{symbols[state[i]]} {symbols[state[i+1]]} {symbols[state[i+2]]}")
    print()

In [53]:
# Vérifier un gagnant
def check_winner(state, player):
    for combo in winning_combinations:
        if all(state[pos] == player for pos in combo):
            return True
    return False

In [54]:
# Clé d’état
def state_to_key(state, is_pose_phase):
    state_tuple = tuple(state) + (is_pose_phase,)
    return hashlib.md5(str(state_tuple).encode()).hexdigest()


In [55]:
# Cases adjacentes
def get_adjacent(pos, state):
    adj = []
    forbidden_diagonals = {(1, 3), (3, 1), (1, 5), (5, 1), (3, 7), (7, 3), (7, 5), (5, 7)}
    if pos % 3 > 0: adj.append(pos - 1)
    if pos % 3 < 2: adj.append(pos + 1)
    if pos >= 3: adj.append(pos - 3)
    if pos <= 5: adj.append(pos + 3)
    if pos % 3 > 0 and pos >= 3 and (pos, pos - 4) not in forbidden_diagonals: adj.append(pos - 4)
    if pos % 3 < 2 and pos >= 3 and (pos, pos - 2) not in forbidden_diagonals: adj.append(pos - 2)
    if pos % 3 > 0 and pos <= 5 and (pos, pos + 2) not in forbidden_diagonals: adj.append(pos + 2)
    if pos % 3 < 2 and pos <= 5 and (pos, pos + 4) not in forbidden_diagonals: adj.append(pos + 4)
    return [p for p in adj if 0 <= p < 9 and state[p] == 0]


In [56]:
# Appliquer un mouvement
def apply_move(state, move, player, is_pose_phase=True):
    new_state = state.copy()
    if is_pose_phase:
        if 0 <= move <= 8 and new_state[move] == 0:
            new_state[move] = player
        else:
            raise ValueError("Mouvement invalide dans la phase de pose")
    else:
        old_pos, new_pos = move
        if (new_state[old_pos] == player and new_state[new_pos] == 0 and 
            new_pos in get_adjacent(old_pos, new_state)):
            new_state[old_pos] = 0
            new_state[new_pos] = player
        else:
            raise ValueError("Mouvement invalide dans la phase de déplacement")
    return new_state

In [57]:
# Mouvements possibles
def get_possible_moves(state, player, is_pose_phase=True):
    possible_moves = []
    if is_pose_phase:
        for pos in range(9):
            if state[pos] == 0:
                possible_moves.append(pos)
    else:
        player_positions = [i for i in range(9) if state[i] == player]
        for old_pos in player_positions:
            adj = get_adjacent(old_pos, state)
            for new_pos in adj:
                if state[old_pos] == player and state[new_pos] == 0:
                    possible_moves.append((old_pos, new_pos))
    return possible_moves

In [58]:
# Charger la Q-Table
def load_q_table():
    global q_table
    if os.path.exists(q_table_file):
        q_table = np.load(q_table_file, allow_pickle=True).item()
        print(f"Q-Table chargée avec {len(q_table)} états.")
    else:
        raise FileNotFoundError(f"Le fichier {q_table_file} n'existe pas.")

In [59]:
# Choisir une action avec diagnostic
def choose_action(state, player, is_pose_phase):
    state_key = state_to_key(state, is_pose_phase)
    moves = get_possible_moves(state, player, is_pose_phase)
    if not moves:
        print("Aucun mouvement possible.")
        return None
    if state_key not in q_table:
        print(f"État inconnu : {state}. Choix aléatoire  parmi {moves}")
        return random.choice(moves)
    else:
        q_values = q_table[state_key]
        valid_q_values = {k: v for k, v in q_values.items() if eval(k) in moves}
        if not valid_q_values:
            print(f"Aucune Q-valeur valide pour {state}. Choix aléatoire parmi {moves}")
            return random.choice(moves)
        print(f"État : {state}, Q-valeurs : {valid_q_values}")
        if random.uniform(0, 1) < epsilon:
            move = random.choice(moves)
            print(f"Exploration : {move}")
            return move
        else:
            best_move_str = max(valid_q_values, key=valid_q_values.get)
            move = int(best_move_str) if is_pose_phase else eval(best_move_str)
            print(f"Meilleur mouvement : {move} (Q = {valid_q_values[best_move_str]})")
            return move

In [60]:
# Tour humain (pose)
def human_turn_pose():
    global human_pieces
    while True:
        try:
            print_board()
            pos = int(input("Choisissez une position (0-8) pour poser un pion : "))
            if pos in get_possible_moves(board, 1, True):
                board[:] = apply_move(board, pos, 1, True)
                human_pieces -= 1
                break
            print("Position invalide ou occupée !")
        except ValueError:
            print("Entrez un nombre valide !")

In [61]:
# Tour IA (pose)
def ai_turn_pose():
    global ai_pieces
    state = board.copy()
    move = choose_action(state, 2, True)
    if move is None:
        return False
    print(f"L’IA pose un pion en {move}")
    board[:] = apply_move(board, move, 2, True)
    ai_pieces -= 1
    return True

In [62]:
# Tour humain (déplacement)
def human_turn_move():
    while True:
        try:
            print_board()
            possible_moves = get_possible_moves(board, 1, False)
            if not possible_moves:
                print("Aucun déplacement possible pour vous !")
                return False
            print(f"Mouvements possibles : {possible_moves}")
            old_pos = int(input("Choisissez un pion à déplacer (0-8) : "))
            if board[old_pos] != 1:
                print("Ce n’est pas votre pion !")
                continue
            new_pos = int(input("Choisissez une nouvelle position : "))
            move = (old_pos, new_pos)
            if move in possible_moves:
                board[:] = apply_move(board, move, 1, False)
                break
            print("Mouvement invalide !")
        except ValueError:
            print("Entrez un nombre valide !")
    return True

In [63]:
# Tour IA (déplacement)
def ai_turn_move():
    possible_moves = get_possible_moves(board, 2, False)
    if not possible_moves:
        print("Aucun déplacement possible pour l’IA !")
        return False
    state = board.copy()
    move = choose_action(state, 2, False)
    if move is None:
        return False
    print(f"L’IA déplace un pion de {move[0]} à {move[1]}")
    board[:] = apply_move(board, move, 2, False)
    return True

In [64]:
# Jeu
def play_game():
    global human_pieces, ai_pieces
    load_q_table()
    print("Bienvenue au jeu ! Phase de pose.")
    while human_pieces > 0 or ai_pieces > 0:
        human_turn_pose()
        print_board()
        if check_winner(board, 1):
            print("Vous avez gagné !")
            return
        if ai_pieces > 0:
            if not ai_turn_pose():
                print("Match nul (aucun mouvement possible pour l’IA) !")
                return
            print_board()
            if check_winner(board, 2):
                print("L’IA a gagné !")
                return
    print("Phase de déplacement commencée !")
    move_count = 0
    max_moves = 50
    while move_count < max_moves:
        if not human_turn_move():
            print("Match nul (aucun mouvement possible) !")
            break
        print_board()
        if check_winner(board, 1):
            print("Vous avez gagné !")
            return
        if not ai_turn_move():
            print("Match nul (aucun mouvement possible) !")
            break
        print_board()
        if check_winner(board, 2):
            print("L’IA a gagné !")
            return
        move_count += 1
    if move_count >= max_moves:
        print("Match nul (limite de coups atteinte) !")
